# Quick SLM — 08 · SFT diagnostics

`07_sft_eval` says **how much** the model gets right. This notebook says **why**
it gets the rest wrong, and writes a single JSON you can hand to someone (or to
a model) for analysis without shipping the corpus.

It is built around the one question that decides what v2 should do:

| explanation | evidence | v2 response |
|---|---|---|
| **composition** | thin cells fail, fat cells pass | rebalance the corpus. Cheap. |
| **capacity** | the model fails data it was *trained on* | build a bigger model. Expensive. |
| **memorisation** | training accuracy high, held-out low | more variety, not more volume |

Held-out accuracy alone cannot separate these — they predict the same numbers.
What separates them is grading a sample of **training** examples too, which is
why section 7 exists and why it costs extra GPU time. A model that fails data it
was fit to has not been out-competed by the corpus mix; it has run out of room.

It also answers *why the corpus deduplicated so hard*, per cell, by measuring how
repetitive the teacher actually was.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
!pip -q install 'transformers>=4.44' 'tokenizers>=0.19' datasketch tqdm accelerate bitsandbytes


## 3. Locate the repository and check framework support

In [ ]:
import sys, json
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/quick-slm/code')
framework_dir = REPO_DIR / 'framework'
assert framework_dir.is_dir(), (
    f'{framework_dir} not found. The tree moved from code/src to code/framework; '
    'upload framework/ and training/v1/ and delete the stale code/src.'
)
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))

from v1.quick_slm_trainer.support import require_framework
require_framework('v1', REPO_DIR)

from v1.quick_slm_trainer.paths import Layout
from v1.quick_slm_trainer.config import sft_v1

DRIVE_ROOT = Path('/content/drive/MyDrive/quick-slm')
layout = Layout(drive_root=DRIVE_ROOT)
cfg = sft_v1()
print('framework OK; ctx', cfg.data.ctx)

## 4. Rebuild the corpus and the held-out split

Both the validated set and the deduplicated set are kept: the difference between
them is the dedup forensics, and it is not recoverable after the fact.

The rebuilt split is checked against the count recorded at packing time. A
mismatch is a hard stop — it would mean measuring on data the model trained on.

In [ ]:
from v1.quick_slm_trainer.sft import load_and_validate
from v1.quick_slm_trainer.sft.corpus import category_histogram, paired_integrity
from v1.quick_slm_trainer.sft.dedup import dedup_examples
from v1.quick_slm_trainer.sft.pack import split_examples

validated, val_stats, _ = load_and_validate(layout, cfg.sft)
deduped = dedup_examples(validated, threshold=cfg.sft.dedup_jaccard,
                         num_perm=cfg.sft.minhash_perms, progress=True)

corpus = list(deduped)          # 04b ran with REBALANCE = False
train_ex, val_ex = split_examples(corpus, val_fraction=cfg.sft.val_fraction)

# The rebuilt split must be the split that was packed. If it is not, the model
# may have trained on what is graded below and every number here is worthless.
#
# Checking the val count alone does not establish that. `split_examples` keys
# ungrouped examples by their enumeration index, so adding one example anywhere
# shifts every later key, reshuffles the whole split, and leaves
# round(N * val_fraction) unchanged. The corpus size is the sharp check.
recorded = json.loads(layout.sft_stats_path.read_text())

def _rec(*path):
    node = recorded
    for k in path:
        if not isinstance(node, dict) or k not in node:
            return None
        node = node[k]
    return node

checks = [
    ('corpus', len(corpus),   _rec('after_rebalance', 'examples') or _rec('after_dedup', 'examples')),
    ('train',  len(train_ex), _rec('pack', 'train', 'examples_in')),
    ('val',    len(val_ex),   _rec('pack', 'val', 'examples_in')),
]
known = [(n, got, want) for n, got, want in checks if want is not None]
assert known, (
    f'corpus_stats.json records no counts to check against (top-level keys: '
    f'{sorted(recorded)}). Re-run 04b section 8 with PACK = True.'
)
mismatched = [(n, got, want) for n, got, want in known if got != want]
assert not mismatched, (
    'the rebuilt corpus is NOT the one that was packed -- '
    + '; '.join(f'{n}: rebuilt {got:,}, pack recorded {want:,}' for n, got, want in mismatched)
    + '. The held-out split therefore differs from the one training used. Re-run '
      '04b with PACK = True, then re-run 05, before trusting anything below.'
)
for n, got, _ in known:
    print(f'  {n:<7}{got:>8,}  matches the pack')

n_pairs, broken = paired_integrity(val_ex)
print(f'\nvalidated {len(validated):,} -> deduped {len(corpus):,}')
print(f'val counterfactual pairs {n_pairs}  broken {len(broken)}')
print('val by category:', category_histogram(val_ex))


## 5. Load the checkpoint

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

STEP = None   # e.g. 899; None -> sft_final/

CKPT = layout.sft_final_dir if STEP is None else layout.sft_ckpt_dir / f'step_{STEP:07d}'
assert CKPT.is_dir(), f'{CKPT} not found'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if device == 'cuda' else torch.float32

tok = AutoTokenizer.from_pretrained(str(layout.tokenizer_dir))
model = AutoModelForCausalLM.from_pretrained(str(CKPT), dtype=dtype).to(device).eval()
tok.padding_side = 'left'     # right padding detaches the prompt from the continuation
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
print('loaded', CKPT.name)

## 6. Generate on the held-out split

Greedy. The question is what the model believes, not what it can be sampled into.

In [ ]:
from tqdm.auto import tqdm
from v1.quick_slm_trainer.template import render_prompt

BATCH   = 32
MAX_NEW = 192

def generate(examples, desc):
    out = []
    for i in tqdm(range(0, len(examples), BATCH), desc=desc):
        chunk = examples[i:i + BATCH]
        enc = tok([render_prompt(e) for e in chunk], return_tensors='pt',
                  padding=True, truncation=True,
                  max_length=cfg.data.ctx - MAX_NEW).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        # Slice by prompt length, not by string search: the tokenizer does not
        # round-trip whitespace exactly and a search would drop leading tokens.
        cut = enc['input_ids'].shape[1]
        out.extend(tok.decode(g[cut:], skip_special_tokens=True) for g in gen)
    return out

val_texts = generate(val_ex, 'val')
print(len(val_texts), 'held-out generations')

## 7. Generate on a sample of **training** examples

This is the control that separates capacity from composition, and it is the
reason this notebook is not just section 6 with more printing.

Sampling is per cell rather than uniform: the memorisation gap needs enough
examples in a cell to mean anything, and a uniform sample would give the thin
cells two examples each and the fat cells hundreds. Groups are kept whole.

In [ ]:
import collections, random
from v1.quick_slm_trainer.sft.diagnose import cell_of, MIN_CELL_N

TRAIN_PER_CELL = 40    # >= MIN_CELL_N, or the gap is not computed for that cell

by_cell = collections.defaultdict(list)
for e in train_ex:
    by_cell[cell_of(e)].append(e)

rng = random.Random(4242)
train_sample = []
for cell in sorted(by_cell):
    pool = by_cell[cell]
    # Sample whole groups so a counterfactual pair never arrives half-included.
    groups = collections.defaultdict(list)
    for e in pool:
        groups[e.meta.get('group') or id(e)].append(e)
    keys = sorted(groups, key=str)
    rng.shuffle(keys)
    picked = []
    for k in keys:
        if len(picked) >= TRAIN_PER_CELL:
            break
        picked.extend(groups[k])
    train_sample.extend(picked)

print(f'{len(train_sample):,} training examples across {len(by_cell)} cells')
print('cells below the floor (gap will not be computed):',
      [c for c in sorted(by_cell) if len(by_cell[c]) < MIN_CELL_N])

train_texts = generate(train_sample, 'train')

## 8. Grade both and build the report

In [ ]:
from v1.quick_slm_trainer.sft.grade import grade, reference_calls, summarise
from v1.quick_slm_trainer.sft.diagnose import build_report

results       = [grade(e, t) for e, t in zip(val_ex, val_texts)]
train_results = [grade(e, t) for e, t in zip(train_sample, train_texts)]

print(summarise(results).table())

report = build_report(
    results=results,
    train_examples=train_ex,
    validated=validated,
    kept=corpus,
    train_results=train_results,
    meta={
        'checkpoint': str(CKPT),
        'n_validated': len(validated),
        'n_deduped': len(corpus),
        'n_train': len(train_ex),
        'n_val': len(val_ex),
        'val_reject_reasons': val_stats.to_dict(),
        'packed': recorded.get('pack'),
        'greedy': True,
        'max_new_tokens': MAX_NEW,
    },
)

## 9. Model-graded evaluation

The oracle in section 8 is authoritative wherever a reference call exists and
its arguments are checkable, and blind everywhere else. Three places matter:

- **refusals** — `answer.text` is excluded from comparison, so `answer("banana")`
  scores identical to a real decline. All 41 held-out examples.
- **traps** — worse. A trap whose correct behaviour is *"ask which city"* is
  satisfied by `answer("it is sunny")`. All 34.
- **`<think>`** — checked for length and markup leakage only, so the
  counterfactual category's actual reasoning step is invisible.

A judge fills those in. It never overrides the oracle, and the pair metric is
never judged — that is ground truth computed from the state, and routing it
through a language model would only add noise to the sharpest number here.

Two controls make the scores auditable. **Calibration** scores a balanced sample
of cases the oracle already decided and reports agreement; below 80 % the other
dimensions are noise rather than measurement. And the judge is Gemma 4, which
also wrote the corpus — self-preference bias points toward flattering a student
that imitated it, so judge scores are reported beside the oracle's numbers and
never blended into them.

Skip this section if you only want the oracle numbers; nothing below depends on it.

In [ ]:
JUDGE = True        # set False to skip the judge entirely
TRIAGE_LIMIT = 150  # wrong-call triage is a sample, not a census
CALIBRATION_N = 60  # balanced; half already-correct, half already-wrong

judge_summary = None
if JUDGE:
    from transformers import BitsAndBytesConfig
    from v1.quick_slm_trainer.sft import judge as JD

    GEMMA_MODEL = 'google/gemma-4-31B-it-qat-q4_0-unquantized'   # same judge as 06

    _q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                            bnb_4bit_compute_dtype=torch.bfloat16,
                            bnb_4bit_use_double_quant=True)
    print(f'loading judge: {GEMMA_MODEL} (4-bit nf4)')
    jtok = AutoTokenizer.from_pretrained(GEMMA_MODEL)
    jmodel = AutoModelForCausalLM.from_pretrained(
        GEMMA_MODEL, quantization_config=_q, device_map='auto').eval()

    def ask(prompt: str) -> str:
        msgs = [{'role': 'user', 'content': prompt}]
        ids = jtok.apply_chat_template(msgs, add_generation_prompt=True,
                                       return_tensors='pt').to(jmodel.device)
        with torch.no_grad():
            out = jmodel.generate(ids, max_new_tokens=64, do_sample=False,
                                  pad_token_id=jtok.eos_token_id)
        return jtok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

    # Calibration first: if the judge cannot match the oracle where the oracle is
    # right, there is no point paying for the rest.
    plan = JD.calibration_targets(results, n=CALIBRATION_N) + \
           JD.targets(results, triage_limit=TRIAGE_LIMIT)
    print(f'judging {len(plan)} cases '
          f'({sum(1 for d, _ in plan if d == JD.CALIBRATION)} of them calibration)')

    verdicts = JD.judge_all(results, ask, plan=plan, progress=lambda it: tqdm(it, desc='judge'))
    judge_summary = JD.summarise(verdicts, results)
    print()
    print('\n'.join(JD.report_lines(judge_summary)))

    report['judge'] = judge_summary
    report['judge_verdicts'] = [
        {'dimension': v.dimension, 'index': v.index, 'category': v.category,
         'score': v.score, 'reason': v.reason}
        for v in verdicts
    ]
else:
    print('JUDGE is False -- oracle numbers only. Refusals and traps are then '
          'graded on call choice alone, which is close to no information.')

## 10. The headline reading

Everything below is in the JSON too. This cell just puts the load-bearing parts
where you can see them without opening the file.

In [ ]:
m, v = report['memorisation'], report['volume_vs_accuracy']
print('=' * 64)
print('VERDICT :', m.get('verdict'))
print('VOLUME  :', v.get('reading'), '(rho', v.get('rho'), ')')
print('=' * 64)
if m.get('cells'):
    print(f"train mean {m['train_mean']:.1%}   held-out mean {m['val_mean']:.1%}"
          f"   median gap {m['median_gap']:+.1%}")

d = report['default_call']
print(f"\nwhen wrong ({d['n_wrong']:,} times) it called "
      f"{d['top_call']} in {d['top_share']:.0%} of them")

c = report['think_call_coupling']
if c['considered']:
    print(f"reasoning named a different tool than it called: "
          f"{c['decoupled']}/{c['considered']} ({c['decoupled_rate']:.1%})")
print(f"reasoning held the answer the call dropped: {c['think_had_answer_call_did_not']}")

print('\nworst cells (held-out correct rate, non-thin):')
worst = sorted((r for r in report['cells'] if not r['thin'] and r['correct_rate'] is not None),
               key=lambda r: r['correct_rate'])[:8]
for r in worst:
    tr = f"{r['train_correct_rate']:.0%}" if r['train_correct_rate'] is not None else '  -'
    print(f"  {r['category']:<22}{r['subtype']:<24}n_train {r['n_train']:>6,}"
          f"   val {r['correct_rate']:>5.0%}   train {tr:>4}")

print('\nworst specifications (counterfactual):')
for r in report['pairs']['by_spec'][:6]:
    print(f"  {r['spec_id']:<24}pairs {r['pairs']:>4}   grounded {r['grounded_rate']:>5.0%}")

print('\nheaviest deduplication (validated -> kept):')
for r in report.get('dedup', [])[:6]:
    print(f"  {r['category']:<22}{r['subtype']:<24}{r['n_validated']:>6,} -> {r['n_kept']:>6,}"
          f"  ({r['kept_rate']:>4.0%})  largest identical cluster {r['largest_cluster']:>4}")

## 11. Save

Two files. The **JSON** is the one to send: it is compact and self-contained. The
**JSONL** keeps every raw generation on Drive so any follow-up question can be
answered without paying for the GPU again — which is worth more than it sounds,
because the question you want next is usually not the one the report answered.

In [ ]:
name = CKPT.name
out_json  = layout.sft_dir / f'diagnostics_{name}.json'
out_jsonl = layout.sft_dir / f'generations_{name}.jsonl'

out_json.write_text(json.dumps(report, indent=2, default=str))

with out_jsonl.open('w') as fh:
    for split, exs, txts in (('val', val_ex, val_texts), ('train', train_sample, train_texts)):
        for e, t in zip(exs, txts):
            fh.write(json.dumps({
                'split': split,
                'category': e.category,
                'subtype': e.meta.get('subtype', ''),
                'spec_id': e.meta.get('spec_id'),
                'group': e.meta.get('group'),
                'branch': e.meta.get('branch'),
                'user': e.turns[0].text,
                'state': e.state,
                'expected': reference_calls(e),
                'generated': t,
            }, default=str) + '\n')

mb = out_json.stat().st_size / 1e6
print(f'wrote {out_json}  ({mb:.2f} MB)   <- send this one')
print(f'wrote {out_jsonl} ({out_jsonl.stat().st_size/1e6:.1f} MB)  <- keep for follow-ups')
if mb > 5:
    print('\nWARNING: the JSON is large. Trim report["failures"] before sending.')